
# Text summarization
*Résumé automatique de texte à l’aide de méthodes de NLP*

# Contexte
Avec l’explosion des contenus textuels sur le web, les entreprises, les chercheurs et les institutions sont confrontés à une surcharge d’information. Pour améliorer l’efficacité de la lecture et faciliter la prise de décision, le développement d’outils capables de générer des résumés pertinents et concis est devenu essentiel.

Ce projet s’inscrit dans cette optique : concevoir un système de résumé automatique de texte capable de fournir des synthèses fiables et compréhensibles, en un temps réduit.

# Objectif
L’objectif principal est de développer un modèle de NLP capable de générer un résumé automatique à partir d’un texte donné.

Le système devra :
2. Générer des résumés extractifs ou abstraits selon les besoins
4. Être potentiellement intégrable dans une interface web simple

# Tâches à réaliser

### A. Résumé Extractif
- Utilisation de modèles pré-entraînés (CamemBERT, RoBERTa)

### B. Résumé Abstractive
- Utilisation de modèles pré-entraînés (T5, BART)

## 5. Déploiement
- Création d’une application simple (Streamlit)
- Interface de saisie de texte et retour du résumé


# Méthode Extractive

In [9]:
!pip install transformers sentencepiece torch sentence-transformers nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 86.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [17]:
from transformers import CamembertTokenizer, CamembertModel, RobertaTokenizer, RobertaModel, BartTokenizer, BartForConditionalGeneration, T5Tokenizer, T5ForConditionalGeneration
import torch
import numpy as np
from nltk import sent_tokenize
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

### Modèle CamemBERT (Français)

In [2]:
def camembert_extractive_summary(text, num_sentences=3):
    # Tokenizer et modèle
    tokenizer = CamembertTokenizer.from_pretrained("camembert-base")
    model = CamembertModel.from_pretrained("camembert-base")

    # Découpage en phrases
    sentences = [s.strip() for s in text.split('.') if len(s) > 0]

    # Encodage et embeddings
    embeddings = []
    for sent in sentences:
        inputs = tokenizer(sent, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        embeddings.append(outputs.last_hidden_state.mean(dim=1).squeeze().numpy())

    # Matrice de similarité
    sim_matrix = np.zeros((len(sentences), len(sentences)))
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                sim_matrix[i][j] = np.dot(embeddings[i], embeddings[j]) / (
                    np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[j]))

    # Scores des phrases
    scores = np.array([sim_matrix[i].sum() for i in range(len(sentences))])

    # Sélection des meilleures phrases
    top_sentences = scores.argsort()[-num_sentences:][::-1]
    summary = '. '.join([sentences[i] for i in sorted(top_sentences)]) + '.'

    return summary

#### Exemple d'utilisation

In [28]:
text = """
La beauté de la nature est quelque chose qui ne pourra jamais être entièrement
capturée par des mots. Chaque arbre, chaque montagne, chaque rivière a sa propre
histoire, une histoire qui est murmurée par le vent et racontée par le bruissement des feuilles.
Lorsque nous prenons le temps de ralentir et de vraiment apprécier le monde qui nous entoure, nous
commençons à voir à quel point tout est interconnecté. Les oiseaux, les animaux,
les plantes et la terre elle-même—ils jouent tous un rôle dans un équilibre délicat
qui soutient la vie. Il est important de se rappeler que nous ne sommes qu'une petite
partie de cette grande tapisserie et que chaque action que nous entreprenons a un impact
sur le monde dans lequel nous vivons.
"""


In [29]:
print(camembert_extractive_summary(text, 2))

La beauté de la nature est quelque chose qui ne pourra jamais être entièrement 
capturée par des mots. Les oiseaux, les animaux, 
les plantes et la terre elle-même—ils jouent tous un rôle dans un équilibre délicat 
qui soutient la vie.


### Modele RoBRTa (Anglais)

In [12]:

def roberta_extractive_summary(text, num_sentences=3):
    # Charger RoBERTa
    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    model = RobertaModel.from_pretrained('roberta-base')

    # Tokeniser les phrases avec NLTK
    sentences = sent_tokenize(text)

    # Générer les embeddings des phrases
    embeddings = []
    for sent in sentences:
        inputs = tokenizer(sent, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        embeddings.append(outputs.last_hidden_state.mean(dim=1).squeeze().numpy())

    # Calculer les scores de similarité entre phrases
    sim_matrix = np.zeros((len(sentences), len(sentences)))
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                sim_matrix[i][j] = np.dot(embeddings[i], embeddings[j]) / (
                    np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[j]))

    # Classer les phrases par importance
    scores = np.array([sim_matrix[i].sum() for i in range(len(sentences))])
    top_sentences = scores.argsort()[-num_sentences:][::-1]

    # Générer le résumé
    summary = ' '.join([sentences[i] for i in sorted(top_sentences)])
    return summary


#### Exemple d'utilisation

In [25]:
# Exemple
text = """
Deep learning (also known as deep structured learning) is part of a
broader family of machine learning methods based on artificial neural networks with
representation learning. Learning can be supervised, semi-supervised or unsupervised.
Deep-learning architectures such as deep neural networks, deep belief networks, deep reinforcement learning,
recurrent neural networks and convolutional neural networks have been applied to
fields including computer vision, speech recognition, natural language processing,
machine translation, bioinformatics, drug design, medical image analysis, material
inspection and board game programs, where they have produced results comparable to
and in some cases surpassing human expert performance. Artificial neural networks
(ANNs) were inspired by information processing and distributed communication nodes
in biological systems. ANNs have various differences from biological brains. Specifically,
neural networks tend to be static and symbolic, while the biological brain of most living organisms
is dynamic (plastic) and analogue. The adjective "deep" in deep learning refers to the use of multiple
layers in the network. Early work showed that a linear perceptron cannot be a universal classifier,
but that a network with a nonpolynomial activation function with one hidden layer of unbounded width can.
Deep learning is a modern variation which is concerned with an unbounded number of layers of bounded size,
which permits practical application and optimized implementation, while retaining theoretical universality
under mild conditions. In deep learning the layers are also permitted to be heterogeneous and to deviate widely
from biologically informed connectionist models, for the sake of efficiency, trainability and understandability,
whence the structured part.
"""


In [27]:
print(roberta_extractive_summary(text, 4))

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Deep learning (also known as deep structured learning) is part of a
broader family of machine learning methods based on artificial neural networks with
representation learning. Specifically,
neural networks tend to be static and symbolic, while the biological brain of most living organisms
is dynamic (plastic) and analogue. The adjective "deep" in deep learning refers to the use of multiple
layers in the network. In deep learning the layers are also permitted to be heterogeneous and to deviate widely
from biologically informed connectionist models, for the sake of efficiency, trainability and understandability,
whence the structured part.


# Méthode Abstractive

#### Modèle T5 (plguillou/t5-base-fr-sum-cnndm) pour le frafrançais

In [18]:
def t5_french_summary(text, max_length=150):
    # Charger le modèle et le tokenizer
    model_name = "plguillou/t5-base-fr-sum-cnndm"
    tokenizer = T5Tokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name)

    # Préparer l'input avec la tâche de summarization
    input_text = "résumer: " + text

    # Tokenization et génération
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=30,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    # Décoder et retourner le résultat
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


#### Exemple d'utilisation

In [20]:
text = """
La Tour Eiffel est une tour de fer puddlé de 324 mètres de hauteur située à Paris.
Construite par Gustave Eiffel pour l'Exposition universelle de 1889, elle est devenue un symbole de la France.
Initialement critiquée par les artistes de l'époque, elle attire aujourd'hui près de 7 millions de visiteurs annuels.
Son entretien nécessite 60 tonnes de peinture tous les 7 ans pour la protéger de la corrosion.
"""


In [21]:
print(t5_french_summary(text, max_length=100))

La Tour Eiffel est une tour de fer puddlé de 324 mètres de hauteur située à Paris. Construite par Gustave Eiffel pour l'Exposition universelle de 1889, elle est devenue un symbole de la France.


#### Modèle BART (facebook/bart-large-cnn) pour la llangue anglaise

In [22]:
def bart_english_summary(text, max_length=130, min_length=30):
    # Charger le modèle et tokenizer BART pré-entraîné pour le summarization
    model_name = "facebook/bart-large-cnn"
    tokenizer = BartTokenizer.from_pretrained(model_name)
    model = BartForConditionalGeneration.from_pretrained(model_name)

    # Préparation de l'input
    inputs = tokenizer([text], max_length=1024, truncation=True, return_tensors="pt")

    # Génération du résumé
    summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=max_length,
        min_length=min_length,
        length_penalty=2.0,
        early_stopping=True
    )

    # Décodage du résultat
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

#### Exemple d'utilisation

In [23]:

text = """
The COVID-19 pandemic has caused unprecedented global disruption.
First identified in Wuhan, China in December 2019, the virus spread rapidly worldwide.
Governments implemented lockdowns and travel restrictions to contain the outbreak.
Vaccine development occurred at record speed, with first doses administered in late 2020.
The pandemic highlighted global inequalities in healthcare access and economic resilience.
Its long-term impacts on work, education, and international relations continue to unfold.
"""


In [24]:
bart_english_summary(text, max_length=150)

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

'First identified in Wuhan, China in December 2019, the virus spread rapidly worldwide. Vaccine development occurred at record speed, with first doses administered in late 2020.'